# CHAPTER 5 梯度下降算法：如何对损失函数进行优化？

你是否还记得我在第二章节末尾中留下的小小的疑问？漫无目的地增加折叠次数事实上是无法让一根直线去拟合任意的曲线的。在人类的世界，我们会有意识地去改变他的形状。但是机器没有意识，如何让机器从数据中**学习**，是这一章的重要课题。

从监督学习的角度讲，我们在训练模型的过程中，实际上就是在寻找一组参数，它能够真实反映（或者尽可能逼近）输入输出之间的关系。**学习**算法每收集一次输入输出，就调整一次参数，使得输入数据可以更好地去预测相应的输出。如果对于这些训练数据，得到的预测结果效果还不错，我们就会希望它在未来面对未知的输入时能够更好地进行预测。

上面的这一段话事实上就是监督学习的核心所在，神经网络的学习过程事实上也是这样的框架，希望这部分能给你一个直观的感受，下面我们将一步步推进，了解神经网络是怎样学习的。

## 5.1 梯度下降算法（GD）

### 5.1.1 为什么使用梯度下降法？

在上一章中，我们介绍了如何从极大似然的角度推导损失函数。以线性回归为例，MSE 损失函数为：

$$\mathcal{L}(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \mathbf{w}^T\mathbf{x}_i)^2$$

我们目标是找到使 $\mathcal{L}(\mathbf{w})$ 最小的 $\mathbf{w}^*$。

**为什么不直接求解解析解？** 理论上，令梯度 $\nabla_{\mathbf{w}}\mathcal{L} = \mathbf{0}$ 可以直接解出：

$$\mathbf{w}^* = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y} \quad \text{（正规方程）}$$

但在实际工程中，这条路几乎走不通：

1. **计算量巨大**：矩阵求逆的复杂度为 $O(d^3)$（$d$ 为特征维度）。当 $d$ 很大（如深度学习中动辄百万级参数），求逆完全不可行。
2. **矩阵不可逆**：当特征之间存在共线性（$\mathbf{X}^T\mathbf{X}$ 不满秩），或者 $n < d$（样本数小于特征数），矩阵不可逆，正规方程直接失效。
3. **非凸优化**：神经网络引入非线性激活函数后，损失函数不再是凸函数，梯度为零的点可能是鞍点或局部极小值，解析求解没有意义。

因此，我们采用**迭代优化**的思路——**梯度下降法**：从一个初始点出发，沿着负梯度方向一步步逼近最小值。


### 5.1.2 梯度下降法原理

#### 梯度的几何直觉

对于函数 $f(\theta)$，梯度 $\nabla f(\theta)$ 是一个向量，指向**函数值增长最快**的方向，其模长等于该方向的方向导数。

<div align="center">
<img src="../attachment/图5-1.jpg" width="520">
</div>

---

#### 数学表述

设损失函数为 $\mathcal{L}(\boldsymbol{\theta})$，其中 $\boldsymbol{\theta} \in \mathbb{R}^d$ 是所有待优化参数的集合。

**梯度的定义**：损失函数对参数 $\boldsymbol{\theta}$ 的梯度为各偏导数组成的向量

$$\boxed{\nabla_{\boldsymbol{\theta}}\mathcal{L}(\boldsymbol{\theta}) = \begin{bmatrix} \frac{\partial \mathcal{L}}{\partial \theta_1} & \frac{\partial \mathcal{L}}{\partial \theta_2} & \cdots & \frac{\partial \mathcal{L}}{\partial \theta_d} \end{bmatrix}^T}$$

**参数更新公式**：

$$\boxed{\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \cdot \nabla_{\boldsymbol{\theta}}\mathcal{L}(\boldsymbol{\theta}_t)}$$

其中：
- $t$：迭代步数（$t = 0, 1, 2, \ldots$）
- $\eta > 0$：**学习率（Learning Rate）**，控制每一步的步长
- $\nabla_{\boldsymbol{\theta}}\mathcal{L}(\boldsymbol{\theta}_t)$：当前参数下的梯度

---



#### 批量梯度下降（BGD）——使用全部数据

**批量梯度下降法（Batch Gradient Descent, BGD）** 每次迭代使用**全部 $n$ 个样本**计算损失和梯度：

$$\boxed{\mathcal{L}_{\text{BGD}}(\boldsymbol{\theta}) = \frac{1}{n}\sum_{i=1}^{n} \mathcal{L}_i(\boldsymbol{\theta})}$$

$$\boxed{\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \cdot \frac{1}{n}\sum_{i=1}^{n} \nabla_{\boldsymbol{\theta}}\mathcal{L}_i(\boldsymbol{\theta}_t)}$$

---


#### 以线性回归为例

对于线性回归模型 $y = wx + b$（为简化，考虑单变量），MSE 损失函数为：

$$\mathcal{L}(w, b) = \frac{1}{n}\sum_{i=1}^{n}(y_i - (wx_i + b))^2$$

梯度计算（分别对 $w$ 和 $b$ 求偏导）：

$$\boxed{\frac{\partial \mathcal{L}}{\partial w} = -\frac{2}{n}\sum_{i=1}^{n} x_i \cdot (y_i - (wx_i + b))}$$

$$\boxed{\frac{\partial \mathcal{L}}{\partial b} = -\frac{2}{n}\sum_{i=1}^{n} (y_i - (wx_i + b))}$$

BGD 的参数更新：

$$w_{t+1} = w_t - \eta \cdot \frac{\partial \mathcal{L}}{\partial w}\Big|_{w_t, b_t}$$

$$b_{t+1} = b_t - \eta \cdot \frac{\partial \mathcal{L}}{\partial b}\Big|_{w_t, b_t}$$

下图展示了梯度下降法在线性回归中的迭代过程。我们将参数空间中的等高线图与数据空间中的直线拟合并列展示，你可以清晰地看到每一步参数更新如何让拟合直线逐步逼近数据。


In [8]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter
from mpl_toolkits.mplot3d import Axes3D

# ============================================================
# 设置中文字体
# ============================================================
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 生成线性回归数据
# ============================================================
np.random.seed(42)
n = 30
x = np.linspace(-3, 3, n)
true_w, true_b = 2.5, 1.2
noise = np.random.randn(n) * 2.0
y = true_w * x + true_b + noise

# ============================================================
# 损失函数 & 梯度
# ============================================================
def mse_loss(w, b, x, y):
    pred = w * x + b
    return np.mean((y - pred) ** 2)

def gradient(w, b, x, y):
    pred = w * x + b
    err = y - pred
    dw = -2 * np.mean(x * err)
    db = -2 * np.mean(err)
    return dw, db

# ============================================================
# BGD 迭代记录
# ============================================================
w_init, b_init = -1.0, -2.0   # 初始参数（离真实值较远）
lr = 0.08
steps = 60

w_hist, b_hist = [w_init], [b_init]
loss_hist = [mse_loss(w_init, b_init, x, y)]

w_cur, b_cur = w_init, b_init
for t in range(steps):
    dw, db = gradient(w_cur, b_cur, x, y)
    w_cur = w_cur - lr * dw
    b_cur = b_cur - lr * db
    w_hist.append(w_cur)
    b_hist.append(b_cur)
    loss_hist.append(mse_loss(w_cur, b_cur, x, y))

w_hist = np.array(w_hist)
b_hist = np.array(b_hist)
loss_hist = np.array(loss_hist)

# ============================================================
# 构建等高线网格
# ============================================================
W_grid = np.linspace(-2, 4, 200)
B_grid = np.linspace(-3, 3, 200)
WW, BB = np.meshgrid(W_grid, B_grid)
ZZ = np.zeros_like(WW)
for i in range(len(W_grid)):
    for j in range(len(B_grid)):
        ZZ[j, i] = mse_loss(W_grid[i], B_grid[j], x, y)

# ============================================================
# 绘制 GIF（双面板：等高线 + 数据拟合）
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5))

def animate(frame):
    ax1.clear(); ax2.clear()

    # --- Panel A: 损失等高线 + 参数轨迹 ---
    levels = np.logspace(np.log10(ZZ.min()), np.log10(ZZ.max()), 30)
    ax1.contour(WW, BB, ZZ, levels=levels, colors='#B0BEC5',
                linewidths=0.5, alpha=0.6)
    cf = ax1.contourf(WW, BB, ZZ, levels=levels, cmap='YlOrRd', alpha=0.85)
    ax1.plot(w_hist[:frame+1], b_hist[:frame+1], '-o', color='#2C3E50',
             markersize=3, linewidth=2, zorder=5)
    ax1.scatter([true_w], [true_b], marker='*', color='#E74C3C',
                s=300, zorder=10, edgecolors='white', linewidth=1.5)
    ax1.annotate(f'★ 生成数据集的真实参数\nw={true_w}, b={true_b}',
                 xy=(true_w, true_b), fontsize=10, fontweight='bold',
                 color='#E74C3C', xytext=(true_w + 0.8, true_b + 0.5),
                 arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.8))

    ax1.set_xlabel('$w$ (斜率)', fontsize=13)
    ax1.set_ylabel('$b$ (截距)', fontsize=13)
    ax1.set_title(f'参数空间 · 损失函数等高线\n'
                  f'迭代 {frame}/{steps}  |  Loss = {loss_hist[frame]:.3f}',
                  fontsize=13, fontweight='bold')
    ax1.set_xlim(-2, 4); ax1.set_ylim(-3, 3)

    # --- Panel B: 数据点 + 拟合线 ---
    ax2.scatter(x, y, c='#3498DB', s=55, alpha=0.7,
                edgecolors='white', linewidth=0.8, zorder=5,
                label=f'训练数据 (n={n})')
    x_line = np.linspace(-3.5, 3.5, 200)

    # 真实直线
    ax2.plot(x_line, true_w * x_line + true_b, '--', color='#95A5A6',
             linewidth=2, alpha=0.7, label=f'真实: y={true_w}x+{true_b}')

    # 当前拟合直线
    y_pred = w_hist[frame] * x_line + b_hist[frame]
    ax2.plot(x_line, y_pred, '-', color='#E74C3C', linewidth=3,
             label=fr'拟合: $y = {w_hist[frame]:.2f}x + {b_hist[frame]:.2f}$')

    # 残差连线
    y_cur = w_hist[frame] * x + b_hist[frame]
    for xi, yi, yci in zip(x, y, y_cur):
        ax2.plot([xi, xi], [yi, yci], color='#E74C3C', alpha=0.25, linewidth=0.6)

    ax2.set_xlabel('$x$', fontsize=13); ax2.set_ylabel('$y$', fontsize=13)
    ax2.set_title(f'数据空间 · 拟合直线\n'
                  fr'$w_{{{frame}}}={w_hist[frame]:.3f},\; b_{{{frame}}}={b_hist[frame]:.3f}$',
                  fontsize=13, fontweight='bold')
    ax2.legend(fontsize=9, loc='upper left')
    ax2.set_xlim(-3.5, 3.5); ax2.set_ylim(y.min() - 2, y.max() + 2)

    fig.suptitle('梯度下降法（BGD）',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    return []

ani = animation.FuncAnimation(fig, animate, frames=steps + 1,
                               interval=200, blit=False, repeat=True)

# 保存为 GIF
gif_path = '../attachment/GD_BGD_animation.gif'
ani.save(gif_path, writer=PillowWriter(fps=5), dpi=120)
print(f'GIF 已保存至: {gif_path}')
plt.close()
print(f'最终参数: w = {w_hist[-1]:.4f}, b = {b_hist[-1]:.4f}')
print(f'真实参数: w = {true_w}, b = {true_b}')
print(f'最终 Loss: {loss_hist[-1]:.4f}')

GIF 已保存至: ../attachment/GD_BGD_animation.gif
最终参数: w = 2.1603, b = 0.8236
真实参数: w = 2.5, b = 1.2
最终 Loss: 2.7620


## 5.2 随机梯度下降法（SGD）

### 5.2.1 BGD 的计算瓶颈

BGD 每次参数更新需要**遍历全部 $n$ 个样本**来计算梯度：

$$\nabla_{\boldsymbol{\theta}}\mathcal{L}(\boldsymbol{\theta}) = \frac{1}{n}\sum_{i=1}^{n} \nabla_{\boldsymbol{\theta}}\mathcal{L}_i(\boldsymbol{\theta})$$

当数据集规模 $n$ 达到百万甚至十亿级别时（这在深度学习中很常见），每次更新都要扫一遍全量数据，计算成本巨大。更何况，梯度下降需要数百甚至数千次迭代——整体开销不可接受。

### 5.2.2 SGD 的核心思想

**随机梯度下降法（Stochastic Gradient Descent, SGD）** 做出了一个大胆的简化：**每次只随机取一个样本来估计梯度**。

$$\boxed{\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \cdot \nabla_{\boldsymbol{\theta}}\mathcal{L}_{i_t}(\boldsymbol{\theta}_t)}$$

其中 $i_t \in \{1, 2, \ldots, n\}$ 是在第 $t$ 步**随机选取**的样本索引。


---

### 5.2.3 SGD 的迭代过程

以线性回归 $y = wx + b$ 为例。在第 $t$ 步，随机选取样本 $(x_{i_t}, y_{i_t})$：

$$\boxed{\mathcal{L}_{i_t}(w, b) = \big(y_{i_t} - (w x_{i_t} + b)\big)^2}$$

单样本梯度：

$$\frac{\partial \mathcal{L}_{i_t}}{\partial w} = -2 x_{i_t} \cdot \big(y_{i_t} - (w x_{i_t} + b)\big)$$

$$\frac{\partial \mathcal{L}_{i_t}}{\partial b} = -2 \cdot \big(y_{i_t} - (w x_{i_t} + b)\big)$$

---

### 5.2.4 SGD 与 BGD 的对比

> **SGD 的震荡特性**：由于每一步只用了一个样本，梯度方向在每步都可能大幅变化。每次迭代，SGD 面临的"等高线"都是不同的（因为损失函数 $\mathcal{L}_i$ 随样本 $i$ 变化）。这导致参数更新路径不再像 BGD 那样平滑，而是在"锯齿"中前进。但正是这种噪声，有时反而能帮助 SGD 跳出局部最优——这是 BGD 做不到的。

---

### 5.2.5 学习率衰减

SGD 的梯度估计永远不会完全"收敛"——即使到达最优参数附近，单个样本的梯度也不会真正归零（因为噪声的梯度不等于零）。这使得 SGD 会在最优解附近**持续震荡**。

为解决这个问题，通常采用**学习率衰减**策略：

$$\eta_t = \frac{\eta_0}{1 + \alpha \cdot t} \quad \text{或} \quad \eta_t = \eta_0 \cdot \gamma^{\lfloor t / \tau \rfloor}$$

- 训练初期：较大的学习率快速推进
- 训练后期：逐渐减小步长，在最优解附近稳定下来

下图展示了 SGD 在每次迭代中，参数更新路径以及损失函数的逐轮变化。注意观察单样本带来的梯度噪声如何让参数路径产生震荡，而整体趋势仍然向着最优解靠近。


In [12]:
# ============================================================
# SGD 可视化 V2: 每次迭代的等高线随选取的样本而变化
# 核心洞察: 单样本损失 L_i(w,b) = (y_i - w*x_i - b)^2
#          其等高线是平行线族, 谷底为直线 b = y_i - w*x_i
#          不同 x_i 的样本 → 谷底朝向不同 → SGD 来回震荡
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# --- 数据 ---
np.random.seed(42)
n = 30
x_data = np.linspace(-3, 3, n)
true_w, true_b = 2.5, 1.2
noise = np.random.randn(n) * 2.0
y_data = true_w * x_data + true_b + noise

# --- 全量损失 & 单样本损失 ---
def mse_full(w, b):
    return np.mean((y_data - (w * x_data + b)) ** 2)

def grad_one(w, b, xi, yi):
    err = yi - (w * xi + b)
    return -2 * xi * err, -2 * err

# --- SGD 迭代 ---
w_init, b_init = -1.0, -2.0
lr = 0.04
steps = 150

rng = np.random.RandomState(123)
indices = rng.randint(0, n, size=steps)

w_hist, b_hist = [w_init], [b_init]
loss_full_hist = [mse_full(w_init, b_init)]

w_cur, b_cur = w_init, b_init
for t in range(steps):
    i = indices[t]
    dw, db = grad_one(w_cur, b_cur, x_data[i], y_data[i])
    w_cur -= lr * dw
    b_cur -= lr * db
    w_hist.append(w_cur)
    b_hist.append(b_cur)
    loss_full_hist.append(mse_full(w_cur, b_cur))

w_hist = np.array(w_hist)
b_hist = np.array(b_hist)
loss_full_hist = np.array(loss_full_hist)

# ============================================================
# 预计算所有单样本的损失网格（每个样本对应一组"平行线"等高线）
# L_k(w,b) = (y_k - w*x_k - b)^2
# 谷底线: b = y_k - w * x_k
# ============================================================
grid_size = 120
W_grid = np.linspace(-2, 4, grid_size)
B_grid = np.linspace(-3, 3, grid_size)
WW, BB = np.meshgrid(W_grid, B_grid)

# 预计算 n 个单样本损失网格
ZZ_each = np.zeros((n, grid_size, grid_size))
for k in range(n):
    ZZ_each[k] = (y_data[k] - WW * x_data[k] - BB) ** 2

# 预计算全量损失网格（作为参考背景）
ZZ_full = np.zeros((grid_size, grid_size))
for i_w in range(grid_size):
    for j_b in range(grid_size):
        ZZ_full[j_b, i_w] = mse_full(W_grid[i_w], B_grid[j_b])

# ============================================================
# 全局颜色范围（统一 colorbar）
# ============================================================
# 取所有单样本损失在网格上的 1%-99% 分位作为公共 color range
all_vals = ZZ_each.ravel()
vmin = np.percentile(all_vals[all_vals > 0], 1)
vmax = np.percentile(all_vals, 95)
levels_fixed = np.logspace(np.log10(vmin), np.log10(vmax), 20)

# ============================================================
# 绘制 GIF（三面板）
# Panel A: 当前样本的损失等高线（不断变化！）+ 梯度箭头（与等高线垂直！）
# Panel B: 全量损失下降曲线
# Panel C: 数据空间 + 当前样本高亮
# ============================================================
fig = plt.figure(figsize=(22, 7))
gs = fig.add_gridspec(1, 3, width_ratios=[1.2, 1, 1])
ax1 = fig.add_subplot(gs[0])   # 等高线（主面板，稍大）
ax2 = fig.add_subplot(gs[1])   # 损失曲线
ax3 = fig.add_subplot(gs[2])   # 数据空间

def animate_sgd(frame):
    ax1.clear(); ax2.clear(); ax3.clear()

    # 确定当前帧使用的样本（等高线和梯度都用这个样本）
    if frame == 0:
        cur_idx = indices[0]   # 初始帧：显示第一个要用的样本
    else:
        cur_idx = indices[frame - 1]  # 刚用完的样本

    ZZ_current = ZZ_each[cur_idx]

    # ================================================================
    # Panel A: 变化的等高线 —— SGD 的核心特征
    # ================================================================
    # 当前单样本的损失等高线（固定 color range，便于跨帧对比）
    cf = ax1.contourf(WW, BB, ZZ_current, levels=levels_fixed,
                      cmap='YlOrRd', alpha=0.85, extend='both')
    ax1.contour(WW, BB, ZZ_current, levels=levels_fixed,
                colors='#B0BEC5', linewidths=0.3, alpha=0.4)

    # 当前样本的谷底线（损失为 0 的直线）: b = y_i - w * x_i
    w_line = np.linspace(-2, 4, 200)
    b_valley = y_data[cur_idx] - w_line * x_data[cur_idx]
    # 裁剪到可视范围
    mask = (b_valley >= -3) & (b_valley <= 3)
    ax1.plot(w_line[mask], b_valley[mask], '--', color='#27AE60',
             linewidth=2.5, alpha=0.85,
             label=f'样本 #{cur_idx} 谷底: $b={y_data[cur_idx]:.1f} - {x_data[cur_idx]:.1f}w$')

    # 梯度方向箭头 —— 与等高线使用同一样本 cur_idx，保证梯度 ⊥ 等高线
    if frame < steps:
        w_cur, b_cur = w_hist[frame], b_hist[frame]
        dw, db = grad_one(w_cur, b_cur, x_data[cur_idx], y_data[cur_idx])
        # 负梯度方向
        arrow_len = 0.5
        grad_norm = np.sqrt(dw**2 + db**2)
        if grad_norm > 1e-8:
            dw_n = -dw / grad_norm * arrow_len
            db_n = -db / grad_norm * arrow_len
            ax1.arrow(w_cur, b_cur, dw_n, db_n,
                      head_width=0.12, head_length=0.18,
                      fc='#2980B9', ec='#2980B9', linewidth=2.5,
                      zorder=15, alpha=0.9)

    # 参数轨迹
    # 半透明显示过去轨迹
    for k in range(max(0, frame - 20), frame):
        alpha = 0.15 + 0.6 * (k - max(0, frame - 20)) / max(1, min(20, frame))
        ax1.plot(w_hist[k:k+2], b_hist[k:k+2], '-', color='#2C3E50',
                 linewidth=1.0, alpha=alpha, zorder=4)
    # 当前点高亮
    ax1.scatter(w_hist[frame], b_hist[frame], color='#2C3E50', s=100,
                zorder=12, edgecolors='white', linewidth=2)

    # 生成数据集的真实参数标记
    ax1.scatter([true_w], [true_b], marker='*', color='#E74C3C',
                s=300, zorder=10, edgecolors='white', linewidth=1.5)
    ax1.annotate(f'★ 生成数据集的真实参数\nw={true_w}, b={true_b}',
                 xy=(true_w, true_b), fontsize=9, fontweight='bold',
                 color='#E74C3C', xytext=(true_w + 0.8, true_b + 0.5),
                 arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.5))

    ax1.set_xlabel('$w$ (斜率)', fontsize=12)
    ax1.set_ylabel('$b$ (截距)', fontsize=12)
    ax1.set_title(f'当前等高线 = 样本 #{cur_idx} 的损失 $\mathcal{{L}}_{{{cur_idx}}}$\n'
                  f'(梯度 ⊥ 等高线)  迭代 {frame}/{steps}',
                  fontsize=12, fontweight='bold')
    ax1.legend(fontsize=8.5, loc='lower right')
    ax1.set_xlim(-2, 4); ax1.set_ylim(-3, 3)

    # ================================================================
    # Panel B: 全量损失 vs 单样本损失
    # ================================================================
    # 全量损失（真实指标）
    ax2.plot(loss_full_hist[:frame+1], '-', color='#2C3E50', linewidth=2.0,
             label='全量数据 MSE（真实指标）')
    if frame > 0:
        ax2.scatter([frame], [loss_full_hist[frame]], color='#2C3E50',
                    s=50, zorder=5)

    # 单样本损失序列（每步用不同样本计算，剧烈跳动）
    loss_each_step = []
    for t in range(frame):
        idx_t = indices[t]
        w_t, b_t = w_hist[t+1], b_hist[t+1]
        loss_each_step.append((y_data[idx_t] - (w_t * x_data[idx_t] + b_t)) ** 2)
    if len(loss_each_step) > 0:
        ax2.plot(range(1, frame+1), loss_each_step, '-', color='#E74C3C',
                 linewidth=0.8, alpha=0.5, label='单样本损失（剧烈跳动）')

    ax2.set_xlabel('迭代次数', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.set_title(f'损失曲线\n当前全量 Loss = {loss_full_hist[frame]:.4f}',
                  fontsize=12, fontweight='bold')
    ax2.legend(fontsize=8, loc='upper right')
    ax2.set_xlim(0, steps)
    ax2.set_ylim(0, max(loss_full_hist[0], np.max(loss_full_hist[:frame+1])) * 1.15)
    ax2.grid(True, alpha=0.2)

    # ================================================================
    # Panel C: 数据空间 + 当前样本高亮 + 拟合直线
    # ================================================================
    ax3.scatter(x_data, y_data, c='#BDC3C7', s=35, alpha=0.35,
                edgecolors='white', linewidth=0.4, zorder=2)

    if frame > 0 or frame == 0:
        if frame > 0:
            cur_idx = indices[frame - 1]
        ax3.scatter([x_data[cur_idx]], [y_data[cur_idx]], color='#E74C3C',
                    s=220, edgecolors='#2C3E50', linewidth=2.5, zorder=10)

    x_line = np.linspace(-3.5, 3.5, 200)
    # 真实直线
    ax3.plot(x_line, true_w * x_line + true_b, '--', color='#95A5A6',
             linewidth=1.5, alpha=0.5)
    # 拟合直线
    y_pred = w_hist[frame] * x_line + b_hist[frame]
    ax3.plot(x_line, y_pred, '-', color='#E74C3C', linewidth=2.8,
             label=fr'SGD: $y={w_hist[frame]:.2f}x+{b_hist[frame]:.2f}$')

    ax3.set_xlabel('$x$', fontsize=12); ax3.set_ylabel('$y$', fontsize=12)
    ax3.set_title('数据空间 · 当前样本\n(此样本决定了本轮等高线形状)',
                  fontsize=12, fontweight='bold')
    ax3.legend(fontsize=8, loc='upper left')
    ax3.set_xlim(-3.5, 3.5); ax3.set_ylim(y_data.min() - 2, y_data.max() + 2)

    fig.suptitle('随机梯度下降法（SGD）· 每次迭代的损失函数都在变化！',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    return []

ani_sgd = animation.FuncAnimation(fig, animate_sgd, frames=steps + 1,
                                   interval=200, blit=False, repeat=True)

gif_path_sgd = '../attachment/SGD_loss_animation.gif'
ani_sgd.save(gif_path_sgd, writer=PillowWriter(fps=3), dpi=100)
print(f'GIF 已保存至: {gif_path_sgd}')
plt.close()
print(f'SGD 最终参数: w = {w_hist[-1]:.4f}, b = {b_hist[-1]:.4f}')
print(f'真实参数:      w = {true_w}, b = {true_b}')
print(f'最终全量 Loss: {loss_full_hist[-1]:.4f}')
print()

GIF 已保存至: ../attachment/SGD_loss_animation.gif
SGD 最终参数: w = 2.8015, b = 0.5009
真实参数:      w = 2.5, b = 1.2
最终全量 Loss: 4.1845



## 5.3 小批量梯度下降法（Mini-Batch Gradient Descent）

### 5.3.1 BGD 与 SGD 的折中

- **BGD**：梯度精确但计算量大（每次 $n$ 个样本）
- **SGD**：计算快但梯度噪声大（每次 $1$ 个样本），收敛不稳定

自然的想法是：**能不能取一个折中——每次用一小部分样本来估计梯度？**

这就是**小批量梯度下降法（Mini-Batch Gradient Descent, MBGD）** 的核心思想。

---

### 5.3.2 数学表述

在第 $t$ 次迭代中，从数据集中随机采样一个**小批量（Mini-Batch）** $\mathcal{B}_t \subset \{1, 2, \ldots, n\}$，批次大小为 $|\mathcal{B}_t| = m$（通常 $m \ll n$，如 32、64、128）。

$$\boxed{\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \cdot \frac{1}{|\mathcal{B}_t|}\sum_{i \in \mathcal{B}_t} \nabla_{\boldsymbol{\theta}}\mathcal{L}_i(\boldsymbol{\theta}_t)}$$

以线性回归为例：

$$\frac{\partial \mathcal{L}_{\mathcal{B}_t}}{\partial w} = -\frac{2}{|\mathcal{B}_t|}\sum_{i \in \mathcal{B}_t} x_i \cdot \big(y_i - (w x_i + b)\big)$$

$$\frac{\partial \mathcal{L}_{\mathcal{B}_t}}{\partial b} = -\frac{2}{|\mathcal{B}_t|}\sum_{i \in \mathcal{B}_t} \big(y_i - (w x_i + b)\big)$$

---

### 5.3.3 三种方法的对比

| 特性 | BGD | SGD | Mini-Batch GD |
|:---|:---|:---|:---|
| **每次样本数** | $n$（全部） | $1$ | $m$（如 32） |
| **梯度估计质量** | 精确 | 噪声大 | 噪声适中 |
| **单步计算量** | $O(n)$ | $O(1)$ | $O(m)$ |
| **收敛稳定性** | 最稳定 | 震荡大 | 较稳定 |
| **GPU 并行** | 一般 | 差 | ✅ **最佳** |
| **工业界使用** | 几乎不用 | 小规模 | ✅ **主流** |

> **为什么 Mini-Batch GD 是工业界的主流？**
>
> 核心原因在于**硬件效率**。现代 GPU 具有数千个计算核心，天生适合并行处理矩阵运算。当 $m$ 设置为 32、64 或 128 时，恰好能充分利用 GPU 的并行能力——一次前向传播同时计算 $m$ 个样本，梯度也同时求出。而 SGD（$m=1$）则完全浪费了 GPU 的并行优势。

所以说，批大小（Batch Size）也是一个相当重要的超参数，选择良好的批大小可以更好地提高小批量梯度下降法的表现。

---

下图展示了 Mini-Batch GD 的迭代过程。

In [7]:
# ============================================================
# Mini-Batch GD 可视化: 核心体现"每次取一部分数据"
# 高亮当前 batch 的样本, 展示子集梯度如何驱动参数更新
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter
from matplotlib.patches import Patch

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# --- 数据 ---
np.random.seed(42)
n = 30
x = np.linspace(-3, 3, n)
true_w, true_b = 2.5, 1.2
noise = np.random.randn(n) * 2.0
y = true_w * x + true_b + noise

# --- 损失函数 & 梯度 ---
def mse_loss(w, b, x, y):
    return np.mean((y - (w * x + b)) ** 2)

def grad_batch(w, b, x_batch, y_batch):
    err = y_batch - (w * x_batch + b)
    dw = -2 * np.mean(x_batch * err)
    db = -2 * np.mean(err)
    return dw, db

# --- Mini-Batch GD 迭代 ---
w_init, b_init = -1.0, -2.0
lr = 0.06
batch_size = 8         # 关键: 每次只用 8 个样本
steps = 80

rng = np.random.RandomState(456)
batch_indices = [rng.choice(n, size=batch_size, replace=False) for _ in range(steps)]

w_hist_mb, b_hist_mb = [w_init], [b_init]
loss_hist_mb = [mse_loss(w_init, b_init, x, y)]

w_cur, b_cur = w_init, b_init
for t in range(steps):
    bi = batch_indices[t]
    dw, db = grad_batch(w_cur, b_cur, x[bi], y[bi])
    w_cur -= lr * dw
    b_cur -= lr * db
    w_hist_mb.append(w_cur)
    b_hist_mb.append(b_cur)
    loss_hist_mb.append(mse_loss(w_cur, b_cur, x, y))

w_hist_mb = np.array(w_hist_mb)
b_hist_mb = np.array(b_hist_mb)
loss_hist_mb = np.array(loss_hist_mb)

# --- 等高线 ---
W_grid = np.linspace(-2, 4, 150)
B_grid = np.linspace(-3, 3, 150)
WW, BB = np.meshgrid(W_grid, B_grid)
ZZ = np.zeros_like(WW)
for i_w in range(len(W_grid)):
    for j_b in range(len(B_grid)):
        ZZ[j_b, i_w] = mse_loss(W_grid[i_w], B_grid[j_b], x, y)

# --- 三面板 GIF ---
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 6.5))

# 为每个样本分配固定颜色（用于标识 batch 中哪些样本被选中）
sample_colors = plt.cm.tab20(np.linspace(0, 1, n))

def animate_mbgd(frame):
    ax1.clear(); ax2.clear(); ax3.clear()

    # --- Panel A: 参数空间轨迹 ---
    levels = np.logspace(np.log10(ZZ.min()), np.log10(ZZ.max()), 28)
    ax1.contour(WW, BB, ZZ, levels=levels, colors='#B0BEC5',
                linewidths=0.4, alpha=0.5)
    ax1.contourf(WW, BB, ZZ, levels=levels, cmap='YlOrRd', alpha=0.75)
    ax1.plot(w_hist_mb[:frame+1], b_hist_mb[:frame+1], '-',
             color='#2C3E50', linewidth=1.5, alpha=0.9, zorder=4)
    if frame > 0:
        ax1.scatter(w_hist_mb[frame], b_hist_mb[frame], color='#E74C3C',
                    s=50, zorder=6)
    ax1.scatter([true_w], [true_b], marker='*', color='#27AE60',
                s=250, zorder=10, edgecolors='white', linewidth=1.5)
    ax1.set_xlabel('$w$', fontsize=12); ax1.set_ylabel('$b$', fontsize=12)
    ax1.set_title(f'Mini-Batch GD 参数轨迹\n'
                  f'Batch size = {batch_size} | 迭代 {frame}/{steps}',
                  fontsize=12, fontweight='bold')
    ax1.set_xlim(-2, 4); ax1.set_ylim(-3, 3)

    # --- Panel B: 损失下降曲线 ---
    ax2.plot(loss_hist_mb[:frame+1], '-', color='#2C3E50', linewidth=2)
    if frame > 0:
        ax2.scatter([frame], [loss_hist_mb[frame]], color='#E74C3C', s=50, zorder=5)
    # 标注当前 batch 信息
    ax2.set_xlabel('迭代次数', fontsize=12)
    ax2.set_ylabel('全量数据 MSE Loss', fontsize=12)
    ax2.set_title(f'损失下降曲线\nLoss = {loss_hist_mb[frame]:.4f}',
                  fontsize=12, fontweight='bold')
    ax2.set_xlim(0, steps)
    ax2.set_ylim(0, max(loss_hist_mb) * 1.1)
    ax2.grid(True, alpha=0.2)

    # --- Panel C: 数据空间 + 高亮 Batch 样本 ---
    # 先画非 batch 样本（灰色）
    if frame > 0:
        bi = batch_indices[frame - 1]
        mask_in = np.zeros(n, dtype=bool)
        mask_in[bi] = True
        # 灰色: 不在 batch 中的样本
        ax3.scatter(x[~mask_in], y[~mask_in], c='#BDC3C7', s=35, alpha=0.4,
                    edgecolors='white', linewidth=0.4, zorder=2)
        # 彩色高亮: batch 中的样本
        ax3.scatter(x[bi], y[bi], c='#E74C3C', s=120, alpha=0.9,
                    edgecolors='#2C3E50', linewidth=2, zorder=8,
                    label=f'当前 Batch ({batch_size} 个样本)')
    else:
        ax3.scatter(x, y, c='#BDC3C7', s=35, alpha=0.4,
                    edgecolors='white', linewidth=0.4, zorder=2)

    x_line = np.linspace(-3.5, 3.5, 200)
    ax3.plot(x_line, true_w * x_line + true_b, '--', color='#95A5A6',
             linewidth=1.5, alpha=0.5, label=f'真实: y={true_w}x+{true_b}')
    y_pred = w_hist_mb[frame] * x_line + b_hist_mb[frame]
    ax3.plot(x_line, y_pred, '-', color='#E74C3C', linewidth=2.5,
             label=fr'拟合: $y={w_hist_mb[frame]:.2f}x+{b_hist_mb[frame]:.2f}$')

    ax3.set_xlabel('$x$', fontsize=12); ax3.set_ylabel('$y$', fontsize=12)
    ax3.set_title(f'数据空间 · 高亮 Batch 样本\n'
                  f'(灰色样本不参与本轮梯度计算)',
                  fontsize=12, fontweight='bold')
    ax3.legend(fontsize=8, loc='upper left')
    ax3.set_xlim(-3.5, 3.5); ax3.set_ylim(y.min() - 2, y.max() + 2)

    fig.suptitle(f'小批量梯度下降法（Mini-Batch GD）· Batch Size = {batch_size}',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    return []

ani_mbgd = animation.FuncAnimation(fig, animate_mbgd, frames=steps + 1,
                                    interval=200, blit=False, repeat=True)

gif_path_mbgd = '../attachment/MiniBatch_GD_animation.gif'
ani_mbgd.save(gif_path_mbgd, writer=PillowWriter(fps=5), dpi=100)
print(f'GIF 已保存至: {gif_path_mbgd}')
plt.close()
print(f'Mini-Batch GD 最终参数: w = {w_hist_mb[-1]:.4f}, b = {b_hist_mb[-1]:.4f}')
print(f'真实参数:               w = {true_w}, b = {true_b}')
print(f'最终 Loss: {loss_hist_mb[-1]:.4f}')
print(f'Batch size: {batch_size} / {n}')

GIF 已保存至: ../attachment/MiniBatch_GD_animation.gif
Mini-Batch GD 最终参数: w = 2.1699, b = 0.7027
真实参数:               w = 2.5, b = 1.2
最终 Loss: 2.7770
Batch size: 8 / 30


In [10]:
# ============================================================
# 三种方法横向对比: BGD vs SGD vs Mini-Batch GD
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# --- 公共数据 ---
np.random.seed(42)
n = 30
x = np.linspace(-3, 3, n)
true_w, true_b = 2.5, 1.2
noise = np.random.randn(n) * 2.0
y = true_w * x + true_b + noise

def mse_loss(w, b, x, y):
    return np.mean((y - (w * x + b)) ** 2)

def grad_all(w, b, x, y):
    err = y - (w * x + b)
    return -2*np.mean(x*err), -2*np.mean(err)

def grad_one(w, b, xi, yi):
    err = yi - (w*xi + b)
    return -2*xi*err, -2*err

def grad_batch(w, b, xb, yb):
    err = yb - (w*xb + b)
    return -2*np.mean(xb*err), -2*np.mean(err)

# --- 预计算三种方法的轨迹 ---
w0, b0 = -1.0, -2.0
steps = 100
batch_size = 8

# BGD
w_bgd, b_bgd = [w0], [b0]
w, b = w0, b0
for _ in range(steps):
    dw, db = grad_all(w, b, x, y)
    w -= 0.08 * dw; b -= 0.08 * db
    w_bgd.append(w); b_bgd.append(b)

# SGD
rng = np.random.RandomState(123)
sgd_idx = rng.randint(0, n, size=steps)
w_sgd, b_sgd = [w0], [b0]
w, b = w0, b0
for t in range(steps):
    i = sgd_idx[t]
    dw, db = grad_one(w, b, x[i], y[i])
    w -= 0.04 * dw; b -= 0.04 * db
    w_sgd.append(w); b_sgd.append(b)

# MBGD
rng2 = np.random.RandomState(456)
mb_batches = [rng2.choice(n, size=batch_size, replace=False) for _ in range(steps)]
w_mb, b_mb = [w0], [b0]
w, b = w0, b0
for t in range(steps):
    bi = mb_batches[t]
    dw, db = grad_batch(w, b, x[bi], y[bi])
    w -= 0.06 * dw; b -= 0.06 * db
    w_mb.append(w); b_mb.append(b)

# --- 等高线 ---
W_grid = np.linspace(-2, 4, 120)
B_grid = np.linspace(-3, 3, 120)
WW, BB = np.meshgrid(W_grid, B_grid)
ZZ = np.zeros_like(WW)
for i_w in range(len(W_grid)):
    for j_b in range(len(B_grid)):
        ZZ[j_b, i_w] = mse_loss(W_grid[i_w], B_grid[j_b], x, y)
levels = np.logspace(np.log10(ZZ.min()), np.log10(ZZ.max()), 22)

# --- 四面板 GIF: 三种方法轨迹 + 损失对比 ---
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
(ax_bgd, ax_sgd), (ax_mb, ax_loss) = axes

# 限制帧数（让三种方法同步展示）
max_frames = steps + 1

def animate_compare(frame):
    ax_bgd.clear(); ax_sgd.clear(); ax_mb.clear(); ax_loss.clear()

    for ax, w_hist, b_hist, title, color in [
        (ax_bgd, w_bgd, b_bgd, 'BGD（全量数据）', '#2980B9'),
        (ax_sgd, w_sgd, b_sgd, 'SGD（单样本）', '#E74C3C'),
        (ax_mb, w_mb, b_mb, f'Mini-Batch GD（batch={batch_size}）', '#27AE60')
    ]:
        ax.contour(WW, BB, ZZ, levels=levels, colors='#B0BEC5',
                   linewidths=0.3, alpha=0.4)
        ax.contourf(WW, BB, ZZ, levels=levels, cmap='YlOrRd', alpha=0.7)
        ax.plot(w_hist[:frame+1], b_hist[:frame+1], '-', color=color,
                linewidth=2.0, zorder=5)
        if frame > 0:
            ax.scatter(w_hist[frame], b_hist[frame], color=color, s=60,
                       zorder=7, edgecolors='white', linewidth=1)
        ax.scatter([true_w], [true_b], marker='*', color='#2C3E50',
                   s=200, zorder=10, edgecolors='white', linewidth=1)
        ax.set_xlabel('$w$', fontsize=11); ax.set_ylabel('$b$', fontsize=11)
        ax.set_title(title, fontsize=13, fontweight='bold', color=color)
        ax.set_xlim(-2, 4); ax.set_ylim(-3, 3)

    # 损失函数对比
    loss_bgd = [mse_loss(w_bgd[i], b_bgd[i], x, y) for i in range(frame+1)]
    loss_sgd = [mse_loss(w_sgd[i], b_sgd[i], x, y) for i in range(frame+1)]
    loss_mb  = [mse_loss(w_mb[i],  b_mb[i],  x, y) for i in range(frame+1)]

    ax_loss.plot(loss_bgd, '-', color='#2980B9', linewidth=2.0, label='BGD', alpha=0.9)
    ax_loss.plot(loss_sgd, '-', color='#E74C3C', linewidth=1.5, label='SGD', alpha=0.7)
    ax_loss.plot(loss_mb,  '-', color='#27AE60', linewidth=2.0, label=f'MBGD (m={batch_size})', alpha=0.9)
    ax_loss.set_xlabel('迭代次数', fontsize=12)
    ax_loss.set_ylabel('全量数据 MSE Loss', fontsize=12)
    ax_loss.set_title('三种方法损失下降对比', fontsize=14, fontweight='bold')
    ax_loss.legend(fontsize=10, loc='upper right')
    ax_loss.set_xlim(0, steps)
    ax_loss.set_ylim(0, max(mse_loss(w0, b0, x, y), max(loss_sgd)) * 1.05)
    ax_loss.grid(True, alpha=0.2)

    fig.suptitle(f'梯度下降三兄弟 · 迭代对比（第 {frame} 步）',
                 fontsize=17, fontweight='bold', y=1.01)
    plt.tight_layout()
    return []

ani_cmp = animation.FuncAnimation(fig, animate_compare, frames=max_frames,
                                   interval=180, blit=False, repeat=True)

gif_path_cmp = '../attachment/GD_ThreeMethods_Comparison.gif'
ani_cmp.save(gif_path_cmp, writer=PillowWriter(fps=6), dpi=100)
print(f'对比 GIF 已保存至: {gif_path_cmp}')
plt.close()

print('=' * 55)
print('三种方法最终结果对比:')
print(f'  BGD  最终: w={w_bgd[-1]:.4f},  b={b_bgd[-1]:.4f},  Loss={mse_loss(w_bgd[-1], b_bgd[-1], x, y):.4f}')
print(f'  SGD  最终: w={w_sgd[-1]:.4f},  b={b_sgd[-1]:.4f},  Loss={mse_loss(w_sgd[-1], b_sgd[-1], x, y):.4f}')
print(f'  MBGD 最终: w={w_mb[-1]:.4f},  b={b_mb[-1]:.4f},  Loss={mse_loss(w_mb[-1], b_mb[-1], x, y):.4f}')
print(f'  真实参数:  w={true_w},       b={true_b}')
print('=' * 55)

对比 GIF 已保存至: ../attachment/GD_ThreeMethods_Comparison.gif
三种方法最终结果对比:
  BGD  最终: w=2.1603,  b=0.8237,  Loss=2.7620
  SGD  最终: w=2.2772,  b=0.7289,  Loss=2.8148
  MBGD 最终: w=2.1398,  b=0.5221,  Loss=2.8544
  真实参数:  w=2.5,       b=1.2


## 5.4 小结

通过上面的对比图我们可以发现，SGD产生的震动是最大的，再其次是MBGD。实际上，这种震动能够很好地解决陷入局部最优的情况。只不过对于像简单的线性回归问题（凸优化），我们可以采用BGD去解决。因为凸函数不存在局部最优的情况，他只存在一个梯度为零的点。

从梯度下降（GD）到随机梯度下降（SGD），我们减少了巨大的计算任务，但是其不稳定性和震荡性是我们不愿意看到的。这一定程度上会为优化带来不确定性。那么如何对随机梯度下降法进行改进呢？我们下一章继续学习。